In [3]:
import numpy as np;
import random as rnd;
import math;

In [4]:
from tensorflow.keras.datasets import mnist;

## Zelf een NN maken zonder library

We gaan nu zelf een Neuraal NEtwerk implementeren zonder gebruik te maken van libraries voor Neurale Netwerken. Sklearn gebruiken om de mnsit data te laden en numpy gebruiken mag uiteraard wel.

In [5]:
def relu(values: list):
    return np.array([max(0, value) for value in values])


def sigmoid(values: list):
    return np.array([1 / (1 + math.exp(-value)) for value in values])

def softmax(values):
    return np.exp(values) / np.sum(np.exp(values))

def cross_entropy(true, predicted):
    # N = predicted.shape[0]
    # loss = -1 / N * np.sum(true * np.log(predicted) + (1 - true) * np.log(1 - predicted))
    # return loss
    return -np.sum(true * np.log(predicted))

In [6]:
def OneHotEncodeDigit(label: int):
    """
    One hot encode a single one digit label
    """
    arr = np.zeros((10))
    arr[label] = 1
    return arr

def OneHotEncode(arr):
    """
    One hot encode an array of labels, like the train_images
    """
    return np.array([OneHotEncodeDigit(item) for item in arr])

def normalize_and_vectorize_image(image: np.array):
    """
    Turn a image of variable dimesions (must be square) into a vector and normalize values between 0 and 1
    """
    flat = image.flatten()

    maximum = np.max(flat)
    normalised: np.array = flat / maximum

    return normalised.astype(np.half)

In [33]:
# Definition of Neural Network

class Node:
    def __init__(self, weights, bias):
        self.weights = weights
        self.bias = bias

    def run(self, value):
        return np.sum(value * self.weights) + self.bias

    """
    modify the weights weights_detla
    """
    def moddify_weights(self, weights_d):
        self.weights = self.weights + weights_d

    """
    modify the bias by bias_delta
    """
    def modify_bias(self, bias_d):
        self.bias = self.bias + bias_d



class DenseLayer:
    def __init__(self, nodeCount: int, input_length: int, activation_func):
        self.nodes: list[Node] = []
        self.logit_cache = np.empty((0))
        self.pre_activation = np.empty((0))

        for i in range(nodeCount):
            self.nodes.append(
                Node(
                    np.random.standard_normal(input_length),
                    rnd.random()))
            self.activation_func = activation_func

    def run(self, value):
        output = np.zeros(len(self.nodes))
        for i, node in enumerate(self.nodes):
            output[i] = node.run(value)
        self.pre_activation = output
        logits = self.activation_func(output)
        self.logit_cache = logits
        return logits



class NeuralNetwork:
    def __init__(self, *args: DenseLayer):
        self.layers: list[DenseLayer] = args

    def predict(self, values):
        for layer in self.layers:
            values = layer.run(values)
        return values

    def predict_many(self, values):
        predicted = []
        for value in values:
            predicted.append(self.predict(value))
        return np.array(predicted)

    def loss(self, values, labels):
        predicted =  self.predict_many(values)
        print(f"Loss: {cross_entropy(labels, predicted)}")
        return predicted

    def backpropegate(self, desired):
        for layer in reversed(self.layers):
            gradient = compute_output_gradient(layer.logit_cache, desired)
            for i, node in enumerate(layer.nodes):
                weight_d = compute_weight_gradients_delta(layer.logit_cache[i], node.weights, -gradient[i])
                # bias_d = compute_bias_delta(layer.logit_cache[i], node.bias, -gradient)
                node.moddify_weights(weight_d)
                # node.modify_bias(bias_d)
            break


    def train(self, train_x, train_y, test_x, test_y, epochs = 1):
        for epoch in range(epochs):
            for i, x1 in enumerate(train_x):
                self.predict(x1)
                self.backpropegate(train_y[i])

            print("finished epoch")
            self.loss(test_x, test_y)




In [22]:
def relu_derivative(x):
    # return np.array([1 if y > 0 else 0 for y in x])
    return 1 if x > 0 else 0

In [19]:
# Methods for backprop

def compute_output_gradient(predicted, correct):
    return predicted - correct

def compute_weight_gradients_delta(hidden_output, weights_old, desired):
    return - weights_old * hidden_output * relu_derivative(desired)

def compute_bias_delta(hidden_bias, bias_old, result):
    return - bias_old * hidden_bias * relu_derivative(result)

### Data laden

Laad de mnist data zoals je het eerder hebt geladen. Split dedara op in een test/training set. Bepaal zelf op hoeveel je split.

Normaliseer de data op 0-1 ipv de pixelwaardes. Dit heb je ook al eerder gedaan. 

Doe een one hot encoding op de labels van de data.

In [10]:
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

train_images = np.array([normalize_and_vectorize_image(img) for img in train_images])
train_labels = OneHotEncode(train_labels)
test_images = np.array([normalize_and_vectorize_image(img) for img in test_images])
test_labels = OneHotEncode(test_labels)

### Activatie functies

Maak de volgende activatiefuncties: 

    def relu(waardes)

en 

    def softmax(waardes) 

Als het goed is kun je ze uit de vorige opdracht gebruiken. 

### Cross entropy loss

Maak een functie

    def cross_entropy(y_true, y_predicted)

functie om de grootte van loss te kunnen berekenen. We gaan de waarde van de loss gebruiken als validatie output. Het wordt dus aan het einde geprint.

### Initialiseer je NN 

Maak de volgende variabelen:

- Kies hoeveel input nodes je hebt. 
- En hoeveel hidden layer nodes? 
- En hoeveel output nodes? Leg uit waarom je dat hebt gekozen.


Maak voor nu 1 input layer en 1 hidden layer. JE mag dit later uitbreiden

Om de initiele gewichten een random waarde te geven kun je np.random.randn gebruiken. Deze geeft een normale verdeling met waarden die ongeveer liggen tussen -3 en +3.

Om grote getallen te voorkomen kun je deze weer kleiner maken door bijv *0.01 te doen, dan wordt de standaardafwijking 0.01

In [36]:
model = NeuralNetwork(
    DenseLayer(16, 784, relu),
    DenseLayer(10, 16, softmax)
)
predicted = model.loss(train_images[:100], train_labels[:100])
# print("Predictions")
# print(predicted[:5])
model.train(train_images[:5000], train_labels[:5000], test_images[:100], test_labels[:100], 30)

Loss: 3327.9899971999207
finished epoch
Loss: 639.2256815869048
finished epoch
Loss: 637.4463879949305
finished epoch
Loss: 635.6454344232715
finished epoch
Loss: 633.8222562102367
finished epoch
Loss: 631.976265493448
finished epoch
Loss: 630.1068498903044
finished epoch
Loss: 628.2133710817096
finished epoch
Loss: 626.2951632903178
finished epoch
Loss: 624.3515316436083
finished epoch
Loss: 622.3817504110424
finished epoch
Loss: 620.3850611033433
finished epoch
Loss: 618.3606704205913
finished epoch
Loss: 616.3077480342865
finished epoch
Loss: 614.2254241867754
finished epoch
Loss: 612.1127870894651
finished epoch
Loss: 609.968880098967
finished epoch
Loss: 607.7926986477289
finished epoch
Loss: 605.5831869027429
finished epoch
Loss: 603.3392341225111
finished epoch
Loss: 601.0596706785104
finished epoch
Loss: 598.7432637028834
finished epoch
Loss: 596.3887123188136
finished epoch
Loss: 593.9946424039565
finished epoch
Loss: 591.559600830194
finished epoch
Loss: 589.0820491146827
fin

In [37]:
predicted = model.loss(train_images[:100], train_labels[:100])
print("Predictions")
print(predicted[:5])

Loss: 507.0935535153078
Predictions
[[1.37546583e-01 9.55887325e-02 1.71644032e-01 8.26799355e-02
  3.89361876e-22 1.32769952e-01 1.21412026e-01 7.49106313e-02
  1.10497860e-01 7.29502480e-02]
 [1.37546583e-01 9.55887325e-02 1.71644032e-01 8.26799355e-02
  9.16594909e-12 1.32769952e-01 1.21412026e-01 7.49106313e-02
  1.10497860e-01 7.29502480e-02]
 [1.37546583e-01 9.55887325e-02 1.71644032e-01 8.26799355e-02
  1.53301465e-12 1.32769952e-01 1.21412026e-01 7.49106313e-02
  1.10497860e-01 7.29502480e-02]
 [1.37546583e-01 9.55887325e-02 1.71644032e-01 8.26799355e-02
  2.57996971e-18 1.32769952e-01 1.21412026e-01 7.49106313e-02
  1.10497860e-01 7.29502480e-02]
 [1.37546583e-01 9.55887325e-02 1.71644032e-01 8.26799355e-02
  4.69015167e-25 1.32769952e-01 1.21412026e-01 7.49106313e-02
  1.10497860e-01 7.29502480e-02]]


### Forward pass
Maak een forward propagation functie. We returnen ook tussenwaarden (cache) zodat we die voor backpropagation runnen gebruiken.

#### Stap 1

Maak functie def forward()
Als parameters hgebruik de volgende:
- inputdata X
- gewichtsmatrix W1 van de hidden layer nodes
- biasvector b1 van de hidden layer nodes
- gewichtsmatrix W2 van de output layer nodes
- biasvector b2 van de output layer nodes

Bereken eerst de invoer van de hidden layer. Vermenigvuldig de input met de gewichten en tel de bias erbij op.


Hint:

denk aan matrixvermenigvuldiging
de vorm moet kloppen: (samples, features) @ (features, hidden)

#### Stap 2

De uitkomst van stap 1 is de ruwe activatie van de hidden layer. Pas nu de door jou geiplementeerde ReLU-functie hierop toe.

#### Stap 3

Gebruik de output van de hidden layer als input voor de outputlaag. Vermenigvuldig met W2 en tel de bias b2 er bij op. Eigenlijk bijna hetzelfde als wat je in stap 1 hebt gedaan maar met andere dimensies.

#### Stap 4

De output van stap 3 zijn “scores” (logits), nog geen kansen. Zet deze scores om naar kansen met de softmax-functie die je eerder hebt gemaakt.

#### Stap 5


Voor de backpropagation stap heb je straks de tussenwaarden nodig.
Sla alle relevante variabelen op in één object (bijv. tuple)

- input
- hidden pre-activatie
- hidden activatie
- output pre-activatie
- voorspelling

#### Stap 6

return nu (bijvoorbeeld in een tuple vorm) de voorspelling en de object uit Stap 5

### Backward pass

We gaan enkele functies voorbereiden die we in de les hebben gehad om de backpropagation stap uit te voeren.

We beginnen met berekening van de gradient van loss.

Tip: Je kunt de formule uit de slides van deze week gebruiken. 

Maak nu een methode 
        
        def compute_output_gradient(final_prediction, correct_answers)
        
die de output (laag) gradient teruggeeft. Je kunt de formule hierboven gebruiken.


Maak nu methode 


    def compute_output_gradients(hidden_output, output_gradient)

die de gradienten van de output‑gewichten teruggeeft. Voeg aan de output van je methode ook de gradient van bias toe (als tuple bijv.). 

Tip: Je kunt de formule uit de slides van deze week gebruiken.


Maak de methode 

    def compute_hidden_gradient(output_gradient, hidden_to_output_weights)
    
Deze zegt hoe sterk elke hidden neuron bijdraagt aan de fout.

Tip: Je kunt de formule uit de slides van deze week gebruiken.

Schrijf een python functie die afgeleide RELU berekent voor een getal:

    def relu_derivative(x)

Pas het aan zodat het voor een lijst van getallen werkt (eg met numpy)


Maak nu methode 

    def compute_hidden_gradients(hidden_gradient, hidden_raw_gradient, input_data)

De functie moet de gradienten van de hidden-laag gewichten en bias teruggeven. Je hebt de afgeleide van de RELU nodig uit de vorige opgave, want alleen die gewichten doen mee.

Tip: Je kunt de formule uit de slides van deze week gebruiken.

Maak nu de backward propagation nmethode die de bovenstaatde methodes zal gbruiken.

    def backward(y_true, cache, W2):

De parameter cache zal uit de forward pass komen en als het goed is de volgende waardes bevatten:

- input
- hidden pre-activatie
- hidden activatie
- output pre-activatie
- voorspelling

Die kun je dus daaruit unpacken.

Doe nu de volgende stappen achter elkaar:

- Bereken de output gradient (gebruik je eerder gemaakte methode)
- Bereken de output gradients (voor gewichten en bias) (gebruik je eerder gemaakte methode en output gradient)

- Bereken de hidden gradient (gebruik je eerder gemaakte methode)
- Bereken de hidden gradients (voor gewichten en bias) (gebruik je eerder gemaakte methode en hidden gradient)

return de output gradients en de hidden gradients bijvoorbeeld in de vorm van een tuple:

    return dW1, db1, dW2, db2

### De training loop. 

JE mag het volgende stukje code gebruiken om je netwerk te trainen. MAar je mag uiteraard ook zelf een andere opzet maken.


In [ ]:
# bedenk een learning rate die je wilt gebruiken
lr = 0.1

for epoch in range(20):

    # forward pass
    voorspelling, cache = forward(X_train, W1, b1, W2, b2)

    # loss berekenen we wel, maar doen er verder niet echt wat mee behalve printen
    # y_train_oh staat voor voor one hot encoded
    loss = cross_entropy(y_train_oh, voorspelling)

    # backward pass
    dW1, db1, dW2, db2 = backward(y_train_oh, cache, W2)

    # update
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if epoch % 5 == 0:
        preds = np.argmax(voorspelling, axis=1)

        # voor accuracy gebruik je class labels, niet one-hot vectors
        acc = np.mean(preds == y_train)
        print(f"Epoch {epoch}, Loss: {loss:.4f}, Acc: {acc:.3f}")

NameError: name 'forward' is not defined

One hot encoding toepassen (op de labels)

### Experimenteer

Werkt het? Mooi, dan kun je nu zelf experimenteren met verschillende setup parameters en kijken wat het beste werkt.